# Little Steps — Complete Interactive App
This notebook launches the existing Little Steps app from Google Drive while preserving the current SQLite database and media folders.

**Before running:** make sure `MyDrive/LittleSteps/` contains `little_steps.db`, `images/`, and `audio/`. Then choose **Runtime → Run all**.

In [ ]:
# LITTLE STEPS — COMPLETE INTERACTIVE APP
# Google Colab launcher. Keeps your existing database, images, and audio files.

import os, sqlite3, shutil, base64, html, mimetypes, json
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import drive
from google.colab import output as colab_output

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/LittleSteps'
IMAGES = os.path.join(PROJECT, 'images')
AUDIO = os.path.join(PROJECT, 'audio')
DRIVE_DB = os.path.join(PROJECT, 'little_steps.db')
LOCAL_DB = '/content/little_steps_app.db'

for required in (DRIVE_DB, IMAGES, AUDIO):
    if not os.path.exists(required):
        raise FileNotFoundError(f'Required Little Steps item not found: {required}')

shutil.copy2(DRIVE_DB, LOCAL_DB)
db = sqlite3.connect(LOCAL_DB, timeout=30)
db.row_factory = sqlite3.Row
cur = db.cursor()

if not cur.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='activity_steps'").fetchone():
    raise RuntimeError('The database does not contain the activity_steps table.')

columns = [r['name'] for r in cur.execute('PRAGMA table_info(activity_steps)')]
if 'completed' not in columns:
    cur.execute('ALTER TABLE activity_steps ADD COLUMN completed INTEGER DEFAULT 0')
db.commit()

def save_progress():
    db.commit()
    with sqlite3.connect(DRIVE_DB, timeout=30) as drive_db:
        db.backup(drive_db)

def slug(name):
    return '_'.join(str(name).strip().lower().split())

def resolve_media(db_path, folder, expected_names):
    for name in expected_names:
        candidate = os.path.join(folder, name)
        if os.path.isfile(candidate):
            return candidate
    if db_path:
        db_path = str(db_path).strip()
        if os.path.isfile(db_path):
            return db_path
        candidate = os.path.join(folder, os.path.basename(db_path))
        if os.path.isfile(candidate):
            return candidate
    wanted = {n.lower() for n in expected_names}
    try:
        for filename in os.listdir(folder):
            if filename.lower() in wanted:
                candidate = os.path.join(folder, filename)
                if os.path.isfile(candidate):
                    return candidate
    except OSError:
        pass
    return None

def cover_for(name):
    stem = slug(name)
    return resolve_media(None, IMAGES, [
        f'{stem}_cover.png', f'{stem}_cover.jpg', f'{stem}_cover.jpeg',
        f'{stem}_step_1.png', f'{stem}_step_1.jpg', f'{stem}_step_1.jpeg'
    ])

def step_image_for(name, step_number, db_path=None):
    stem = slug(name)
    return resolve_media(db_path, IMAGES, [
        f'{stem}_step_{step_number}.png',
        f'{stem}_step_{step_number}.jpg',
        f'{stem}_step_{step_number}.jpeg'
    ])

def step_audio_for(name, step_number, db_path=None):
    stem = slug(name)
    return resolve_media(db_path, AUDIO, [
        f'{stem}_step_{step_number}.mp3',
        f'{stem}_step_{step_number}.wav',
        f'{stem}_step_{step_number}.m4a',
        f'{stem}_step_{step_number}.ogg'
    ])

def get_activities():
    return cur.execute('''
        SELECT activity_id, activity_name
        FROM activity_steps
        GROUP BY activity_id, activity_name
        ORDER BY activity_id
    ''').fetchall()

def get_steps(activity_id):
    return cur.execute('''
        SELECT step_number, instruction, image_path, audio_path, completed
        FROM activity_steps
        WHERE activity_id=?
        ORDER BY step_number
    ''', (activity_id,)).fetchall()

activities = get_activities()
if not activities:
    raise RuntimeError('No activities were found in activity_steps.')

state = {'activity_id': None, 'name': None, 'step': 1, 'screen': 'home', 'page': 0}
CARDS_PER_PAGE = 3

top_bar_area = widgets.Output()
home_area = widgets.Output()
intro_area = widgets.Output()
lesson_area = widgets.Output()
progress_area = widgets.Output()

GLOBAL_CSS = '''<style>
.ls-3d-btn {
    display: inline-flex; align-items: center; justify-content: center; gap: 8px;
    font-family: inherit; font-weight: 800; border-radius: 16px; cursor: pointer;
    user-select: none; transition: transform 0.08s ease, box-shadow 0.08s ease, filter 0.12s ease;
    text-decoration: none; outline: none;
}
.ls-3d-btn:hover:not(:disabled) { filter: brightness(1.04); }
.ls-3d-btn:active:not(:disabled) { transform: translateY(4px); }
.ls-3d-btn:disabled { opacity: 0.45; cursor: not-allowed; transform: none; }
.ls-btn-teal {
    background: linear-gradient(180deg, #e6f7f4 0%, #c6ece5 100%);
    color: #155e63; border: 2px solid #8fd1c5;
    box-shadow: 0 6px 0 #73b9ad, 0 10px 18px rgba(21, 94, 99, 0.18);
}
.ls-btn-teal:active:not(:disabled) { box-shadow: 0 2px 0 #73b9ad, 0 4px 8px rgba(21, 94, 99, 0.18); }
.ls-btn-green {
    background: linear-gradient(180deg, #48c782 0%, #2fa363 100%);
    color: #ffffff; border: 2px solid #248a52;
    box-shadow: 0 6px 0 #1d7344, 0 10px 18px rgba(29, 115, 68, 0.25);
}
.ls-btn-green:active:not(:disabled) { box-shadow: 0 2px 0 #1d7344, 0 4px 8px rgba(29, 115, 68, 0.25); }
.ls-btn-end {
    background: linear-gradient(180deg, #ff8a65 0%, #f4511e 100%);
    color: #ffffff; border: 2px solid #d84315;
    box-shadow: 0 6px 0 #bf360c, 0 10px 18px rgba(191, 54, 12, 0.24);
}
.ls-btn-end:active:not(:disabled) { box-shadow: 0 2px 0 #bf360c, 0 4px 8px rgba(191, 54, 12, 0.24); }
.ls-btn-icon { width: 54px; height: 50px; font-size: 22px; padding: 0; border-radius: 14px; }
.ls-btn-nav { min-width: 175px; height: 60px; padding: 0 24px; font-size: 21px; }
.ls-btn-start { min-width: 240px; height: 66px; padding: 0 32px; font-size: 25px; }
.ls-btn-arrow { width: 60px; height: 54px; font-size: 28px; padding: 0; }
.ls-btn-card-cta { width: 100%; height: 48px; font-size: 19px; margin-top: 8px; }
.ls-activity-card {
    width: 268px; padding: 14px; background: #ffffff; border: 2px solid #cbe6df;
    border-radius: 22px; box-shadow: 0 7px 0 #b8dcd3, 0 12px 22px rgba(22, 124, 128, 0.12);
    cursor: pointer; transition: transform 0.1s ease, box-shadow 0.1s ease, border-color 0.15s ease;
    display: flex; flex-direction: column; align-items: center; text-align: center;
}
.ls-activity-card:hover { border-color: #38a66a; transform: translateY(-2px); }
.ls-activity-card:active { transform: translateY(4px); box-shadow: 0 2px 0 #b8dcd3, 0 5px 10px rgba(22, 124, 128, 0.12); }
.ls-intro-pic-btn {
    display: inline-block; padding: 8px; background: #ffffff; border: 4px solid #bce4dc;
    border-radius: 26px; box-shadow: 0 8px 0 #9fd3c8, 0 14px 24px rgba(22, 124, 128, 0.15);
    cursor: pointer; transition: transform 0.1s ease, box-shadow 0.1s ease;
}
.ls-intro-pic-btn:hover { border-color: #38a66a; }
.ls-intro-pic-btn:active { transform: translateY(4px); box-shadow: 0 3px 0 #9fd3c8, 0 6px 12px rgba(22, 124, 128, 0.15); }
.ls-lesson-pic-btn {
    display: block; margin: 14px auto; padding: 6px; border: 4px solid #b9e5dc;
    border-radius: 24px; background: #ffffff; max-width: 92%; cursor: pointer;
    box-shadow: 0 7px 0 #9cd1c6, 0 12px 24px rgba(25, 90, 85, 0.14);
    transition: transform 0.08s ease, box-shadow 0.08s ease;
}
.ls-lesson-pic-btn:active { transform: translateY(4px); box-shadow: 0 2px 0 #9cd1c6, 0 5px 10px rgba(25, 90, 85, 0.14); }
</style>'''

brand = widgets.HTML(GLOBAL_CSS + '''
<div style='background:linear-gradient(120deg,#167c80,#38a6a0,#7b70bd); padding:22px 14px;border-radius:22px;color:white;text-align:center;'>
  <div style='font-size:38px'>🌈</div>
  <div style='font-size:40px;font-weight:900'>Little Steps</div>
  <div style='font-size:18px;margin-top:5px'>Every big achievement starts with a little step!</div>
</div>''')

text_size = widgets.Dropdown(options=[('Standard','24px'),('Large','30px'),('Extra large','36px')], value='30px', description='Text:')
high_contrast = widgets.Checkbox(value=False, description='High contrast')
simple_mode = widgets.Checkbox(value=True, description='Simple instructions')
settings_box = widgets.VBox(
    [widgets.HTML("<h3 style='color:#167c80;margin:0 0 8px 0'>Accessibility settings</h3>"),
     widgets.HBox([text_size, high_contrast, simple_mode])],
    layout=widgets.Layout(padding='14px', border='2px solid #cde5df', border_radius='16px')
)
settings_box.layout.display = 'none'

def hide_all_screens():
    home_area.layout.display = 'none'
    intro_area.layout.display = 'none'
    lesson_area.layout.display = 'none'

def open_intro(aid, name):
    state.update(activity_id=int(aid), name=str(name), step=1, screen='intro')
    render_intro()

def begin_activity():
    state['step'] = 1
    state['screen'] = 'lesson'
    render_lesson(play_audio=True)

def end_activity():
    if state['activity_id'] is not None:
        cur.execute(
            'UPDATE activity_steps SET completed=1 WHERE activity_id=? AND step_number=?',
            (state['activity_id'], state['step'])
        )
        db.commit()
        save_progress()
    state.update(activity_id=None, name=None, step=1, page=0, screen='home')
    render_home()

def show_settings():
    settings_box.layout.display = 'none' if settings_box.layout.display != 'none' else 'flex'

def show_progress():
    if progress_area.layout.display != 'none':
        progress_area.layout.display = 'none'
        return
    with progress_area:
        clear_output(wait=True)
        rows = cur.execute('''
            SELECT activity_name, COUNT(*) AS total, COALESCE(SUM(completed), 0) AS done
            FROM activity_steps
            GROUP BY activity_id, activity_name
            ORDER BY activity_id
        ''').fetchall()
        blocks = []
        for r in rows:
            pct = int(r['done'] * 100 / r['total']) if r['total'] else 0
            blocks.append(
                "<div style='padding:14px;margin:10px 0;background:#eff8f5;border-radius:16px;border:1px solid #cbe6df'>"
                + "<b style='font-size:20px;color:#155e63'>" + html.escape(r['activity_name']) + "</b>"
                + "<div style='margin-top:4px;font-size:16px;color:#3f5968'>" + str(r['done']) + " / " + str(r['total']) + " steps completed — " + str(pct) + "%</div>"
                + "<div style='height:14px;background:#d5e6e0;border-radius:10px;margin-top:8px;overflow:hidden'>"
                + "<div style='height:14px;width:" + str(pct) + "%;background:#38a66a;border-radius:10px'></div></div></div>"
            )
        display(HTML("<h2 style='color:#167c80;margin-top:6px'>📊 My Little Steps Progress</h2>" + ''.join(blocks)))
    progress_area.layout.display = 'block'

def handle_top_action(action):
    if action == 'home':
        progress_area.layout.display = 'none'
        state.update(activity_id=None, name=None, step=1, screen='home')
        render_home()
    elif action == 'progress':
        show_progress()
    elif action == 'settings':
        show_settings()

def handle_open_activity(aid, name):
    open_intro(aid, name)

def handle_page_action(direction):
    if direction == 'left':
        state['page'] = max(0, state['page'] - 1)
    elif direction == 'right':
        max_page = max(0, (len(activities) - 1) // CARDS_PER_PAGE)
        state['page'] = min(max_page, state['page'] + 1)
    render_home()

def handle_begin_activity():
    begin_activity()

def handle_nav_action(action):
    if state['activity_id'] is None:
        return
    rows = get_steps(state['activity_id'])
    total = len(rows)
    if action == 'prev':
        if state['step'] > 1:
            state['step'] -= 1
            render_lesson(play_audio=True)
    elif action == 'next':
        if state['step'] < total:
            cur.execute(
                'UPDATE activity_steps SET completed=1 WHERE activity_id=? AND step_number=?',
                (state['activity_id'], state['step'])
            )
            db.commit()
            save_progress()
            state['step'] += 1
            render_lesson(play_audio=True)
    elif action == 'end':
        end_activity()

colab_output.register_callback('little_steps.top_action', handle_top_action)
colab_output.register_callback('little_steps.open_activity', handle_open_activity)
colab_output.register_callback('little_steps.page_action', handle_page_action)
colab_output.register_callback('little_steps.begin_activity', handle_begin_activity)
colab_output.register_callback('little_steps.nav_action', handle_nav_action)

def render_top_bar():
    top_html = (
        "<div style='display:flex;justify-content:flex-end;gap:12px;padding:4px 2px 8px 2px'>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-icon' title='Choose an activity' "
        + "onclick=\"google.colab.kernel.invokeFunction('little_steps.top_action', ['home'], {});\">🏠</button>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-icon' title='My progress' "
        + "onclick=\"google.colab.kernel.invokeFunction('little_steps.top_action', ['progress'], {});\">📊</button>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-icon' title='Accessibility settings' "
        + "onclick=\"google.colab.kernel.invokeFunction('little_steps.top_action', ['settings'], {});\">⚙️</button>"
        + "</div>"
    )
    with top_bar_area:
        clear_output(wait=True)
        display(HTML(top_html))

def make_activity_card_html(aid, name):
    path = cover_for(name)
    if path:
        with open(path, 'rb') as f:
            raw = base64.b64encode(f.read()).decode('ascii')
        mime = mimetypes.guess_type(path)[0] or 'image/png'
        pic = "<img src='data:" + mime + ";base64," + raw + "' style='width:100%;height:210px;object-fit:contain;background:#f5fbfa;border-radius:16px'>"
    else:
        pic = "<div style='width:100%;height:210px;display:flex;align-items:center;justify-content:center;background:#f5fbfa;border-radius:16px;font-size:52px'>🖼️</div>"

    safe_name = html.escape(str(name))
    js_name = json.dumps(str(name))
    onclick_js = "google.colab.kernel.invokeFunction('little_steps.open_activity', [" + str(int(aid)) + ", " + js_name + "], {});"
    onclick_attr = html.escape(onclick_js, quote=True)

    return (
        "<div role='button' tabindex='0' class='ls-activity-card' onclick=\"" + onclick_attr + "\" "
        + "onkeydown=\"if(event.key==='Enter'||event.key===' '){ " + onclick_attr + " }\">"
        + "<div style='width:100%'>" + pic + "</div>"
        + "<div style='font-size:21px;font-weight:800;text-align:center;padding:10px 4px 4px 4px;color:#23445a'>" + safe_name + "</div>"
        + "<div class='ls-3d-btn ls-btn-green ls-btn-card-cta'>Let's learn! →</div>"
        + "</div>"
    )

def render_home():
    state['screen'] = 'home'
    start = state['page'] * CARDS_PER_PAGE
    visible = activities[start:start + CARDS_PER_PAGE]
    cards_html = ''.join(make_activity_card_html(r['activity_id'], r['activity_name']) for r in visible)

    left_disabled = ' disabled' if state['page'] == 0 else ''
    right_disabled = ' disabled' if (start + CARDS_PER_PAGE >= len(activities)) else ''

    home_html = (
        "<div style='text-align:center;padding:18px 8px 10px 8px'>"
        + "<div style='font-size:32px;font-weight:900;color:#235b65'>What would you like to learn today?</div>"
        + "<div style='font-size:19px;color:#617b86;margin-top:6px'>Tap any picture or card to get started.</div>"
        + "</div>"
        + "<div style='display:flex;justify-content:center;align-items:stretch;gap:18px;flex-wrap:wrap;padding:8px'>"
        + cards_html
        + "</div>"
        + "<div style='display:flex;justify-content:center;gap:28px;padding:18px 0 10px 0'>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-arrow' title='Previous activities'" + left_disabled + " "
        + "onclick=\"this.disabled=true; google.colab.kernel.invokeFunction('little_steps.page_action', ['left'], {});\">‹</button>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-arrow' title='More activities'" + right_disabled + " "
        + "onclick=\"this.disabled=true; google.colab.kernel.invokeFunction('little_steps.page_action', ['right'], {});\">›</button>"
        + "</div>"
    )

    hide_all_screens()
    home_area.layout.display = 'block'
    with home_area:
        clear_output(wait=True)
        display(HTML(home_html))

def render_intro():
    hide_all_screens()
    state['screen'] = 'intro'
    intro_area.layout.display = 'block'

    path = cover_for(state['name'])
    safe_name = html.escape(str(state['name']))
    safe_lower = html.escape(str(state['name']).lower())
    start_onclick = "google.colab.kernel.invokeFunction('little_steps.begin_activity', [], {});"

    if path:
        with open(path, 'rb') as f:
            raw = base64.b64encode(f.read()).decode('ascii')
        mime = mimetypes.guess_type(path)[0] or 'image/png'
        image_html = (
            "<button type='button' class='ls-intro-pic-btn' title='Tap picture to start' "
            + "onclick=\"this.disabled=true; " + start_onclick + "\">"
            + "<img src='data:" + mime + ";base64," + raw + "' "
            + "style='display:block;width:min(560px,86vw);max-height:340px;object-fit:contain;border-radius:20px'>"
            + "</button>"
        )
    else:
        image_html = (
            "<button type='button' class='ls-intro-pic-btn' style='padding:40px 60px;font-size:22px;color:#167c80;font-weight:800' "
            + "onclick=\"this.disabled=true; " + start_onclick + "\">Tap here to begin</button>"
        )

    intro_html = (
        "<div style='text-align:center;padding:18px'>"
        + "<div style='font-size:42px;font-weight:900;color:#167c80'>" + safe_name + "</div>"
        + "<div style='font-size:22px;margin:12px 0 16px 0;color:#3f5968'>Today we will learn how to " + safe_lower + ".</div>"
        + image_html
        + "<div style='font-size:19px;margin:16px 0 14px 0;color:#617b86'>Tap the picture or Start when you are ready.</div>"
        + "<div style='display:flex;justify-content:center;padding:6px'>"
        + "<button type='button' class='ls-3d-btn ls-btn-green ls-btn-start' "
        + "onclick=\"this.disabled=true; " + start_onclick + "\">▶ START</button>"
        + "</div></div>"
    )

    with intro_area:
        clear_output(wait=True)
        display(HTML(intro_html))

def render_lesson(play_audio=False):
    state['screen'] = 'lesson'
    hide_all_screens()
    lesson_area.layout.display = 'block'

    rows = get_steps(state['activity_id'])
    row = next((r for r in rows if int(r['step_number']) == int(state['step'])), None)
    if row is None:
        with lesson_area:
            clear_output(wait=True)
            display(HTML('<b>Could not find this lesson step in the database.</b>'))
        return

    total = len(rows)
    bg, fg = ('#101820', '#fff36d') if high_contrast.value else ('#f0faf7', '#23445a')
    title_color = '#fff36d' if high_contrast.value else '#167c80'
    instruction = str(row['instruction'] or 'Follow the picture for this step.')
    image_path = step_image_for(state['name'], state['step'], row['image_path'])
    audio_path = step_audio_for(state['name'], state['step'], row['audio_path'])

    audio_tag = ''
    if audio_path and os.path.isfile(audio_path):
        with open(audio_path, 'rb') as af:
            audio_b64 = base64.b64encode(af.read()).decode('ascii')
        audio_mime = mimetypes.guess_type(audio_path)[0] or 'audio/mpeg'
        autoplay_attr = ' autoplay' if play_audio else ''
        audio_tag = (
            "<audio id='ls-step-audio' preload='auto'" + autoplay_attr + " style='display:none'>"
            + "<source src='data:" + audio_mime + ";base64," + audio_b64 + "' type='" + audio_mime + "'>"
            + "</audio>"
        )

    replay_js = "var a=document.getElementById('ls-step-audio'); if(a){a.pause(); a.currentTime=0; a.play();}"

    if image_path:
        with open(image_path, 'rb') as f:
            image_b64 = base64.b64encode(f.read()).decode('ascii')
        mime = mimetypes.guess_type(image_path)[0] or 'image/png'
        picture_html = (
            "<button type='button' class='ls-lesson-pic-btn' aria-label='Tap picture to hear instruction again' "
            + "onclick=\"" + replay_js + "\">"
            + "<img src='data:" + mime + ";base64," + image_b64 + "' "
            + "style='display:block;width:min(720px,86vw);height:min(430px,54vh);object-fit:contain;border-radius:18px'>"
            + "</button>"
        )
    else:
        picture_html = (
            "<button type='button' class='ls-lesson-pic-btn' style='padding:50px 80px;font-size:20px' "
            + "onclick=\"" + replay_js + "\">Picture unavailable (Tap to hear audio)</button>"
        )

    step_label = '' if simple_mode.value else (
        "<div style='font-size:18px;font-weight:700;opacity:0.85;margin-top:4px;color:" + fg + "'>"
        + 'Step ' + str(state['step']) + ' of ' + str(total)
        + '</div>'
    )

    prev_disabled = ' disabled' if state['step'] <= 1 else ''
    next_btn_html = ''
    if state['step'] < total:
        next_btn_html = (
            "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-nav' "
            + "onclick=\"this.disabled=true; google.colab.kernel.invokeFunction('little_steps.nav_action', ['next'], {});\">"
            + "Next ➡</button>"
        )

    end_btn_class = 'ls-btn-green' if state['step'] == total else 'ls-btn-end'
    end_btn_html = (
        "<button type='button' class='ls-3d-btn " + end_btn_class + " ls-btn-nav' "
        + "onclick=\"this.disabled=true; google.colab.kernel.invokeFunction('little_steps.nav_action', ['end'], {});\">"
        + "✓ End Activity</button>"
    )

    lesson_html = (
        "<div style='background:" + bg + ";color:" + fg + ";padding:20px 18px 26px 18px;border-radius:24px;text-align:center;margin:8px auto;max-width:1020px;border:2px solid #d2ebe4'>"
        + "<div style='font-size:38px;font-weight:900;line-height:1.15;color:" + title_color + "'>" + html.escape(str(state['name'])) + "</div>"
        + step_label
        + "<div style='font-size:" + text_size.value + ";font-weight:800;line-height:1.35;margin:14px auto;max-width:860px'>" + html.escape(instruction) + "</div>"
        + "<div style='font-size:16px;opacity:0.8;margin-bottom:6px'>Tap the picture to hear the instruction again</div>"
        + picture_html
        + audio_tag
        + "<div style='display:flex;justify-content:center;align-items:center;gap:20px;flex-wrap:wrap;margin-top:22px'>"
        + "<button type='button' class='ls-3d-btn ls-btn-teal ls-btn-nav'" + prev_disabled + " "
        + "onclick=\"this.disabled=true; google.colab.kernel.invokeFunction('little_steps.nav_action', ['prev'], {});\">"
        + "⬅ Previous</button>"
        + next_btn_html
        + end_btn_html
        + "</div></div>"
    )

    with lesson_area:
        clear_output(wait=True)
        display(HTML(lesson_html))

def accessibility_change(change):
    if state['screen'] == 'lesson':
        render_lesson(play_audio=False)

text_size.observe(accessibility_change, names='value')
high_contrast.observe(accessibility_change, names='value')
simple_mode.observe(accessibility_change, names='value')

app = widgets.VBox(
    [top_bar_area, brand, settings_box, progress_area, home_area, intro_area, lesson_area],
    layout=widgets.Layout(width='100%', gap='10px')
)
display(app)

intro_area.layout.display = 'none'
lesson_area.layout.display = 'none'
progress_area.layout.display = 'none'

render_top_bar()
render_home()
print('Little Steps is ready. Select an activity card to begin.')